# LoRA fine-tuning and LLM evaluation

**JobReady AI lab.** Run each cell with Shift+Enter. Try the exercises before opening the solutions at the end.

**You will:**
- Count trainable parameters with and without LoRA
- Fine-tune a small model on a formatting task
- Evaluate with a golden set and exact-match

In [ ]:
!pip -q install transformers peft datasets accelerate
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model

## 1. Wrap a model with LoRA

In [ ]:
name = 'google/flan-t5-small'
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)
lora = LoraConfig(r=8, lora_alpha=16, target_modules=['q', 'v'], lora_dropout=0.05, task_type='SEQ_2_SEQ_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # well under 1%

## 2. A small task: turn messy text into a JSON record

In [ ]:
import random, json
random.seed(0)
names = ['Asha', 'Ben', 'Chen', 'Dara', 'Eli', 'Fatima', 'Gopal', 'Hana']; cities = ['Pune', 'Leeds', 'Lagos', 'Dubai', 'Austin']
def example():
    n, c, a = random.choice(names), random.choice(cities), random.randint(18, 60)
    return f'{n} is {a} and lives in {c}', json.dumps({'name': n, 'age': a, 'city': c})
train = [example() for _ in range(400)]; test = [example() for _ in range(40)]
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
for epoch in range(3):
    for i in range(0, len(train), 16):
        batch = train[i:i+16]
        enc = tok([f'Extract JSON: {x}' for x, _ in batch], return_tensors='pt', padding=True)
        lab = tok([y for _, y in batch], return_tensors='pt', padding=True).input_ids
        lab[lab == tok.pad_token_id] = -100
        loss = model(**enc, labels=lab).loss
        opt.zero_grad(); loss.backward(); opt.step()
    print('epoch', epoch, 'loss', round(loss.item(), 3))

## 3. Evaluate on a golden set

In [ ]:
model.eval()
def predict(x):
    out = model.generate(**tok(f'Extract JSON: {x}', return_tensors='pt'), max_new_tokens=40)
    return tok.decode(out[0], skip_special_tokens=True)
valid = exact = 0
for x, y in test:
    p = predict(x)
    try: obj = json.loads(p); valid += 1; exact += obj == json.loads(y)
    except Exception: pass
print(f'valid JSON {valid}/{len(test)}, exact match {exact}/{len(test)}')

## Exercises
1. Run the evaluation *before* fine-tuning (reload the base model). How much did LoRA help?
2. Try r = 4 and r = 16. Record trainable parameters and exact match in a table.

In [ ]:
# your work here

## Solutions
The base model rarely outputs valid JSON in this exact format, while the LoRA model usually gets most records exactly right after 3 epochs. Higher ranks train more parameters, but on a task this simple they rarely add much.